# Onboarding modernized filing stats spreadsheet notebook

## Overview
This notebook generates the data for the onboarding modernized filing stats spreadsheet.

## What it does
- Extracts migration data from COLIN Extract database
- Retrieves filing information from LEAR database
- Merges and exports data to Excel format

## Output
A formatted Excel spreadsheet for filing stats of each comapny.

In [ ]:
%pip install pandas
%pip install sqlalchemy>=2.0
%pip install oracledb
%pip install dotenv
%pip install psycopg2-binary
%pip install openpyxl

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from datetime import datetime
from openpyxl.styles import Alignment

load_dotenv()

COLUMN_NAMES = {
    "corp_num": "Incorporation Number",
    "corp_name": "Company Name",
    "filings": "Filings",
    "filing_date": "Filing Date"
}

CONFIG = {
    'excel_export': {
        'max_column_width': 65,
        'output_dir': os.getenv('EXPORT_OUTPUT_DIR')
    }
}

DATABASE_CONFIG = {
    'lear': {
        'username': os.getenv("DATABASE_LEAR_USERNAME"),
        'password': os.getenv("DATABASE_LEAR_PASSWORD"),
        'host': os.getenv("DATABASE_LEAR_HOST"),
        'port': os.getenv("DATABASE_LEAR_PORT"),
        'name': os.getenv("DATABASE_LEAR_NAME")
    }
}

corp_type_codes = ('BC', 'C', 'ULC', 'CUL', 'CC', 'CCC')


def build_postgres_uri(db_config):
    return f"postgresql://{db_config['username']}:{db_config['password']}@{db_config['host']}:{db_config['port']}/{db_config['name']}"


def create_pg_engines(config_map):
    engines = {}
    for db_key, db_config in config_map.items():
        uri = build_postgres_uri(db_config)
        engine = create_engine(uri)
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        engines[db_key] = engine
        print(f"{db_key.upper()} database engine created and tested successfully.")
    return engines


def fetch_lear_identifiers(engine, legal_types):
    query = f"""
        SELECT DISTINCT
            b.identifier
        FROM businesses b
        JOIN filings f ON b.id = f.business_id
        WHERE
            f.status = 'TOMBSTONE'
            AND b.legal_type IN {legal_types}
        ORDER BY
            b.identifier
        """
    with engine.connect() as conn:
        return pd.read_sql(query, conn)


def fetch_lear_filings(engine, corp_nums, column_names):
    corp_nums_str = ','.join(f"'{num}'" for num in corp_nums)
    query = f"""
        SELECT
            b.identifier AS "{column_names['corp_num']}",
            b.legal_name AS "{column_names['corp_name']}",
            COALESCE(
                STRING_AGG(f.filing_type, ', '), ''
            ) AS "{column_names['filings']}",
            COALESCE(
                STRING_AGG(f.filing_date::text, ', '), ''
            ) AS "{column_names['filing_date']}"
        FROM businesses b
        LEFT JOIN filings f ON b.id = f.business_id
            AND f.source = 'LEAR'
            AND f.status = 'COMPLETED'
        WHERE b.identifier IN ({corp_nums_str})
        GROUP BY b.identifier, b.legal_name, f.filing_type, f.filing_date
        """
    with engine.connect() as conn:
        return pd.read_sql(query, conn)


def auto_adjust_excel_columns(worksheet):
    for column in worksheet.columns:
        max_length = max(
            (len(str(cell.value)) for cell in column if cell.value is not None),
            default=0
        )
        column_letter = column[0].column_letter
        worksheet.column_dimensions[column_letter].width = min(max_length + 3, CONFIG['excel_export']['max_column_width'])

    for row in worksheet.iter_rows(min_row=1, max_row=worksheet.max_row, min_col=1, max_col=worksheet.max_column):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True, vertical='top')


def export_to_excel(df, output_path):
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Sheet1', index=False)
        worksheet = writer.sheets['Sheet1']
        auto_adjust_excel_columns(worksheet)


# Main execution
try:
    engines = create_pg_engines(DATABASE_CONFIG)
    print("All database engines ready for use.")

    lear_df = fetch_lear_identifiers(engines['lear'], corp_type_codes)
    print(f"Total number of migrated businesses: {len(lear_df)}")
    corp_nums = lear_df['identifier'].tolist()

    lear_df = fetch_lear_filings(engines['lear'], corp_nums, COLUMN_NAMES)
    print(f"Total number of filings found: {len(lear_df)}")

    output_file = os.path.join(CONFIG['excel_export']['output_dir'], f"lear_filing_stats_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx")
    export_to_excel(lear_df, output_file)
    print(f"Data exported to Excel file: {output_file}")

except Exception as e:
    print(f"Unexpected error: {e}")
    raise